In [2]:
# Data handling
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt

import os
import warnings
warnings.filterwarnings("ignore")

In [3]:
metadata_full = pd.read_csv("data/seaAD/SEAAD_DFC_RNAseq_final-nuclei_metadata.2026-06-22.csv")

In [4]:
col_list_full = metadata_full.columns.tolist()
col_list_full

['exp_component_name',
 'sample_id',
 'Neurotypical reference',
 'Donor ID',
 'Organism',
 'Brain Region',
 'Sex',
 'Gender',
 'Age at Death',
 'Race (choice=White)',
 'Race (choice=Black/ African American)',
 'Race (choice=Asian)',
 'Race (choice=American Indian/ Alaska Native)',
 'Race (choice=Native Hawaiian or Pacific Islander)',
 'Race (choice=Unknown or unreported)',
 'Race (choice=Other)',
 'specify other race',
 'Hispanic/Latino',
 'Highest level of education',
 'Years of education',
 'PMI',
 'Fresh Brain Weight',
 'Brain pH',
 'Overall AD neuropathological Change',
 'Thal',
 'Braak',
 'CERAD score',
 'Overall CAA Score',
 'Highest Lewy Body Disease',
 'Total Microinfarcts (not observed grossly)',
 'Total microinfarcts in screening sections',
 'Atherosclerosis',
 'Arteriolosclerosis',
 'LATE',
 'Cognitive Status',
 'Last CASI Score',
 'Interval from last CASI in months',
 'Last MMSE Score',
 'Interval from last MMSE in months',
 'Last MOCA Score',
 'Interval from last MOCA in m

In [5]:
metadata_full[:5]

,exp_component_name,sample_id,Neurotypical reference,Donor ID,Organism,Brain Region,Sex,Gender,Age at Death,Race (choice=White),...,Number of reads,Number of UMIs,Genes detected,Doublet score,Fraction mitochondrial UMIs,Severely Affected Donor,Class,Subclass,Supertype,Used in analysis
0,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,90615.0,34882.0,7246,0.072289,0.000602,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_13,True
1,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,22026.0,7869.0,3237,0.000000,0.001271,N,Neuronal: GABAergic,Sst,Sst_19,True
2,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,154515.0,34773.0,7263,0.060241,0.000690,N,Neuronal: Glutamatergic,L6 CT,L6 CT_2,True
3,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,56441.0,12069.0,4914,0.084337,0.000580,N,Non-neuronal and Non-neural,Astrocyte,Astro_2,True
4,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,False,H20.33.016,human,DFC,Female,Female,93.0,Checked,...,129551.0,38153.0,7491,0.048193,0.000970,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_1,True


In [6]:
# Some metadata information
print("Braak stage: ", metadata_full["Braak"].unique())
print("CERAD score: ", metadata_full["CERAD score"].unique())
print("Cognitive status: ", metadata_full["Cognitive Status"].unique())
print("Overall AD neuropathological Change: ", metadata_full["Overall AD neuropathological Change"].unique())

Braak stage:  ['Braak IV' 'Braak VI' 'Braak V' 'Braak III' 'Braak 0' 'Braak II']
CERAD score:  ['Moderate' 'Frequent' 'Absent' 'Sparse']
Cognitive status:  ['Dementia' 'No dementia']
Overall AD neuropathological Change:  ['Intermediate' 'High' 'Low' 'Not AD']


In [12]:
# Cleaning of metadata, drop majority of columns
metadata = metadata_full.drop(metadata_full.loc[:,'Last CASI Score':'Fraction mitochondrial UMIs'].columns, axis=1)
metadata = metadata.drop(metadata.loc[:,'Race (choice=White)':'Years of education'].columns, axis=1)

# Remove columns with no extra information (ex. Brain Region has only DFC)
metadata = metadata.drop(columns=["Neurotypical reference", "Organism", "Brain Region", "Used in analysis"])

In [14]:
# Remaining columns
col_list = metadata.columns.tolist()
col_list

['exp_component_name',
 'sample_id',
 'Donor ID',
 'Sex',
 'Gender',
 'Age at Death',
 'PMI',
 'Fresh Brain Weight',
 'Brain pH',
 'Overall AD neuropathological Change',
 'Thal',
 'Braak',
 'CERAD score',
 'Overall CAA Score',
 'Highest Lewy Body Disease',
 'Total Microinfarcts (not observed grossly)',
 'Total microinfarcts in screening sections',
 'Atherosclerosis',
 'Arteriolosclerosis',
 'LATE',
 'Cognitive Status',
 'Severely Affected Donor',
 'Class',
 'Subclass',
 'Supertype']

In [15]:
metadata[:5]

,exp_component_name,sample_id,Donor ID,Sex,Gender,Age at Death,PMI,Fresh Brain Weight,Brain pH,Overall AD neuropathological Change,...,Total Microinfarcts (not observed grossly),Total microinfarcts in screening sections,Atherosclerosis,Arteriolosclerosis,LATE,Cognitive Status,Severely Affected Donor,Class,Subclass,Supertype
0,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,AAACCCAAGCGTCTGC-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_13
1,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,AAACCCAAGTGGTTAA-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: GABAergic,Sst,Sst_19
2,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,AAACCCACATCCGAAT-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L6 CT,L6 CT_2
3,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,AAACCCACATCTAACG-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Non-neuronal and Non-neural,Astrocyte,Astro_2
4,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,AAACCCAGTATTTCTC-L8TX_210805_01_B03-1153814324,H20.33.016,Female,Female,93.0,7.733333,1242.0,6.8,Intermediate,...,0.0,0.0,Moderate,Severe,Not Identified,Dementia,N,Neuronal: Glutamatergic,L2/3 IT,L2/3 IT_1


In [ ]:
data_path = ".h5ad"
# ensure that the file was properly saved
adata_full = sc.read_h5ad(data_path)
print(f"Successfully read and saved {adata_full.n_obs} cells")

In [ ]:
# Show information about data file
print(adata_full)

In [ ]:
# Show example
print(adata_full.obs.head())

In [ ]:
# Print all available category names you can use for plotting
print(adata_full.obs.columns)

In [ ]:
# Sparsity Check
print(f"Sparse AnnData: {adata_full.n_obs} n_obs x {adata_full.n_vars} n_vars")
sparsity = 1 - adata_full.X.nnz / (adata_full.n_obs * adata_full.n_vars)
print(f"Sparsity: {sparsity:.2%}")